# Install lirary


In [1]:
!pip install -r requirements.txt

     |████████████████████████████████| 12.4 MB 7.0 MB/s eta 0:00:01    |██                              | 788 kB 1.3 MB/s eta 0:00:09
     |████████████████████████████████| 2.3 MB 9.8 MB/s eta 0:00:01
     |████████████████████████████████| 250 kB 10.9 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 32.4 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 8.1 MB/s eta 0:00:01
     |████████████████████████████████| 17.3 MB 13.2 MB/s eta 0:00:01


# Import Libary

In [2]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from itables import show
import itertools

import nest_asyncio
import asyncio
import pandas as pd
import sys

nest_asyncio.apply()  # Patch Jupyter's asyncio loop

async def run_command(
    cmd,
    cwd=None,
    timeout=None,
    print_realtime=False
):
    """Run a shell command asynchronously with high performance.

    Args:
        cmd (str): The shell command to run.
        cwd (str): Optional working directory.
        timeout (float): Timeout in seconds.
        print_realtime (bool): If True, prints stdout/stderr line by line.

    Returns:
        dict: {
            "returncode": int,
            "stdout": str,
            "stderr": str,
            "command": str,
        }
    """
    proc = await asyncio.create_subprocess_shell(
        cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
        cwd=cwd
    )

    stdout, stderr = [], []

    try:
        async def read_stream(stream, buffer, stream_name):
            while True:
                line = await stream.readline()
                if line:
                    decoded = line.decode(errors="replace")
                    buffer.append(decoded)
                    if print_realtime:
                        print(f"[{stream_name}] {decoded}", end="")
                else:
                    break

        await asyncio.wait_for(
            asyncio.gather(
                read_stream(proc.stdout, stdout, "STDOUT"),
                read_stream(proc.stderr, stderr, "STDERR"),
                proc.wait()
            ),
            timeout=timeout
        )

    except asyncio.TimeoutError:
        proc.kill()
        return {
            "returncode": -1,
            "stdout": "".join(stdout),
            "stderr": "".join(stderr) + "\n[Timeout]",
            "command": cmd,
        }

    return {
        "returncode": proc.returncode,
        "stdout": "".join(stdout),
        "stderr": "".join(stderr),
        "command": cmd,
    }

def display_data(df):

    # --- Pivot to compare BRRT vs BRRT_Optimize ---
    pivot_time = df.pivot(index="run", columns="algorithm", values="search_time")
    pivot_length = df.pivot(index="run", columns="algorithm", values="path_length")
    pivot_nodes = df.pivot(index="run", columns="algorithm", values="node_count")
    pivot_success = df.pivot(index="run", columns="algorithm", values="success")
    pivot_idx = df.pivot(index="run", columns="algorithm", values="num_iterations")

    # --- Display deep-inspect comparison table ---


    # show(display_df, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
    # --- Plot line charts for per-run comparison ---
    fig, axs = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

    pivot_time.plot(ax=axs[0], marker='o', title="Search Time per Run")
    axs[0].set_ylabel("Time (s)")
    axs[0].grid(True)

    pivot_length.plot(ax=axs[1], marker='o', title="Path Length per Run")
    axs[1].set_ylabel("Length")
    axs[1].grid(True)

    pivot_nodes.plot(ax=axs[2], marker='o', title="Node Count per Run")
    axs[2].set_ylabel("Nodes")
    axs[2].set_xlabel("Run")
    axs[2].grid(True)


    pivot_idx.plot(ax=axs[3], marker='o', title="Iterations Count per Run")
    axs[3].set_ylabel("Interations")
    axs[3].set_xlabel("Run")
    axs[3].grid(True)

    plt.tight_layout()
    plt.show()
    return df


# Load all files into a list of dictionaries
def summary_statistics(raw_data):
    records = []
    for run_entry in raw_data["results"]:
        run = run_entry["run"]
        for algo_name, result in run_entry["algorithms"].items():
            records.append({
                "run": run,
                "algorithm": algo_name,
                "search_time": result["search_time"],
                "path_length": result["path_length"],
                "node_count": result["node_count"],
                "num_iterations": result.get("num_iterations", 0),
                "success": result["success"],
                "start": result["start"],
                "goal": result["goal"],
            })

    df = pd.DataFrame(records)
    # print(df)
    # --- Display deep-inspect comparison table ---


    agg = df.groupby("algorithm").agg(
        success_rate=("success", "mean"),
        avg_search_time=("search_time", "mean"),
        avg_path_length=("path_length", "mean"),
        avg_node_count=("node_count", "mean"),
        avg_iterations=("num_iterations", "mean")
    ).round(6)
    # show(agg)
    return df,agg

def summary_statistics_single(raw_data):
    records = []
    for run_entry in raw_data["results"]:
        run = run_entry["run"]
        for algo_name, result in run_entry["algorithms"].items():
            records.append({
                "run": run,
                "algorithm": algo_name,
                "search_time": result["search_time"],
                "path_length": result["path_length"],
                "node_count": result["node_count"],
                "num_iterations": result.get("num_iterations", 0),
                "success": result["success"],
                "start": result["start"],
                "goal": result["goal"],
            })

    df = pd.DataFrame(records)
    # print(df)
    # --- Display deep-inspect comparison table ---


    agg = df.groupby("algorithm").agg(
        success_rate=("success", "mean"),
        avg_search_time=("search_time", "mean"),
        avg_path_length=("path_length", "mean"),
        avg_node_count=("node_count", "mean"),
        avg_iterations=("num_iterations", "mean")
    ).round(6)

    sum = df.groupby("algorithm")[["search_time","path_length","node_count","num_iterations"]].sum().round(6)
    # show(agg)
    return df,agg,sum

def load_a_data(output_path):
    with open(output_path, 'r') as f:
        content = json.load(f)
        print("Parameters:", content["parameters"])
        sum_df, agg,sum = summary_statistics_single(content)
        show(sum, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
        show(agg, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
        show (sum_df, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
        display_data(sum_df)
        

# AUTO TURNING

In [ ]:
!bash ./build.sh
!source devel/setup.bash && roslaunch path_finder test_planners.launch

In [3]:
!source devel/setup.bash 

# Generate test parameter

In [1]:
import random
import time
import json
import os

# Define value sets
map_size = 100
epsilon_values = [0.5, 1.0, 2.0]
p1_values = [0.8, 1]
p2_values = [0.8, 0.5, 0.3]
u_p_values = [1,2,3]

# alpha_values = [0, 1, 3, 6, 10]
beta_values = [0.2, 0.5, 0, 1, 4 ]
# gamma_values = [0, 1, 5, 7]
n_rays_values = [8]
weight_grade_values = [1, 2]

max_trials = 20
input_dir = "/home/huy/rrt/experiments/2026_05_21/input/"

# Create input directory if it doesn't exist
if not os.path.exists(input_dir):
    os.makedirs(input_dir)

# Generate max_trials random configurations
for trial in range(1, max_trials + 1):
    # Randomly select one value from each set
    epsilon = random.choice(epsilon_values)
    p1 = random.choice(p1_values)
    p2 = random.choice(p2_values)
    u_p = random.choice(u_p_values)
    alpha = 1
    beta = random.choice(beta_values)
    gamma = beta
    n_rays = random.choice(n_rays_values)
    weight_grade = random.choice(weight_grade_values)
    ray_step_size = epsilon

    print(f"Running trial {trial} with epsilon={epsilon}, p1={p1}, u_p={u_p}, alpha={alpha}, beta={beta}, gamma={gamma}, n_rays={n_rays}, weight_grade={weight_grade}")
    
    environment = f"{epsilon:.2f}_{p1}_{p2}_{u_p}_{round(alpha,2)}_{round(beta,2)}_{round(gamma,2)}_{n_rays}_{weight_grade}_{ray_step_size}"
    
    config = {
        "trial": trial,
        "environment": environment,
        "p1": p1,
        "p2":p2,
        "u_p": u_p,
        "alpha": round(alpha, 2),
        "beta": round(beta, 2),
        "gamma": round(gamma, 2),
        "epsilon": epsilon,
        "n_rays": n_rays,
        "weight_grade": weight_grade,
        "ray_step_size":ray_step_size
    }
    
    input_path = os.path.join(input_dir, f"brrt_{environment}.json")
    with open(input_path, "w") as f:
        json.dump(config, f, indent=4)
        
    time.sleep(0.1)  # Sleep to avoid rapid file creation


Running trial 1 with epsilon=1.0, p1=1, u_p=3, alpha=1, beta=0.5, gamma=0.5, n_rays=8, weight_grade=1
Running trial 2 with epsilon=1.0, p1=0.8, u_p=1, alpha=1, beta=0.2, gamma=0.2, n_rays=8, weight_grade=2
Running trial 3 with epsilon=2.0, p1=1, u_p=2, alpha=1, beta=1, gamma=1, n_rays=8, weight_grade=2
Running trial 4 with epsilon=1.0, p1=1, u_p=1, alpha=1, beta=0.2, gamma=0.2, n_rays=8, weight_grade=2
Running trial 5 with epsilon=0.5, p1=1, u_p=1, alpha=1, beta=0.2, gamma=0.2, n_rays=8, weight_grade=1
Running trial 6 with epsilon=0.5, p1=1, u_p=1, alpha=1, beta=1, gamma=1, n_rays=8, weight_grade=1
Running trial 7 with epsilon=0.5, p1=0.8, u_p=3, alpha=1, beta=0.5, gamma=0.5, n_rays=8, weight_grade=2
Running trial 8 with epsilon=2.0, p1=0.8, u_p=1, alpha=1, beta=1, gamma=1, n_rays=8, weight_grade=2
Running trial 9 with epsilon=2.0, p1=1, u_p=2, alpha=1, beta=0, gamma=0, n_rays=8, weight_grade=2
Running trial 10 with epsilon=2.0, p1=1, u_p=1, alpha=1, beta=0.5, gamma=0.5, n_rays=8, weig

# Run tests

In [ ]:
import os
import time
output_dir =  "/home/huy/rrt/experiments/20250710/output-21/"
input_dir =  "/home/huy/rrt/experiments/20250710/input/"

# create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

files_input = set(os.listdir(input_dir))
files_output = set(os.listdir(output_dir))

only_in_input = files_input - files_output
print("Files in A not in B:")
for f in sorted(only_in_input):
    print(f)
    cmd = f"source /home/huy/rrt/devel/setup.bash && roslaunch path_finder test_planners.launch input_param:={input_dir + f} output_result:={output_dir+f}"
    print(f"Running command: {cmd}")
    ! {cmd}
    time.sleep(1)

Files in A not in B:
brrt_0.50_0.3_0.5_1_7_1_18_2.json
Running command: source /home/huy/rrt/devel/setup.bash && roslaunch path_finder test_planners.launch input_param:=/home/huy/rrt/experiments/20250710/input/brrt_0.50_0.3_0.5_1_7_1_18_2.json output_result:=/home/huy/rrt/experiments/20250710/output-21/brrt_0.50_0.3_0.5_1_7_1_18_2.json
... logging to /home/huy/.ros/log/ef2a0e5a-3091-11f1-b704-e354a8ccdd92/roslaunch-huyvu-pc-34138.log
Checking log directory for disk usage. This may take a while.
Press Ctrl-C to interrupt
Done checking log file disk usage. Usage is <1GB.
]2;/home/huy/rrt/src/path_finder/launch/test_planners.launch
started roslaunch server http://huyvu-pc:43453/

SUMMARY

PARAMETERS
 * /path_finder_node/BRRT/max_iteration: 100000
 * /path_finder_node/BRRT/max_tree_node_nums: 200000
 * /path_finder_node/BRRT/search_time: 1.0
 * /path_finder_node/BRRT/steer_length: 1.0
 * /path_finder_node/BRRT/step: 0.5
 * /path_finder_node/BRRT_Optimize/alpha: 7.0
 * /path_finder_node/B

# Read all files in the directory

In [3]:
import json
import glob
import pandas as pd
def load_a_file(file_path):
    with open(file_path, 'r') as f:
        content = json.load(f)    
        records = []
        for run_entry in content["results"]:
            run = run_entry["run"]
            for algo_name, result in run_entry["algorithms"].items():
                records.append({
                    "run": run,
                    "algorithm": algo_name,
                    "search_time": result["search_time"],
                    "path_length": result["path_length"],
                    "node_count": result["node_count"],
                    "num_iterations": result.get("num_iterations", 1) +1,
                    "success": result["success"],
                    # "start": result["start"],
                    # "goal": result["goal"],
                })

        return pd.DataFrame(records), content["parameters"]
def compare_two_algorithms_success(df):
    # Filter for BRRT and BRRT_Optimize
    brrt_data = df[df['algorithm'] == 'BRRT']
    brrt_opt_data = df[df['algorithm'] == 'BRRT_Optimize']

    # Merge on 'run' to compare success rates
    merged = pd.merge(brrt_data[['run', 'success']], brrt_opt_data[['run', 'success']], on='run', suffixes=('_brrt', '_brrt_opt'))

    # Calculate success rate for each algorithm
    success_rate_brrt = merged['success_brrt'].mean()
    success_rate_brrt_opt = merged['success_brrt_opt'].mean()

    # Filter only successful runs
    df_success = df[df['success'] == True]

    # Count how many algorithms succeeded per run
    success_counts = df_success.groupby('run')['algorithm'].nunique()

    # Keep only runs where both BRRT and BRRT_Optimize succeeded
    valid_runs = success_counts[success_counts == 2].index
    df_valid = df_success[df_success['run'].isin(valid_runs)]

    # Now pivot and compare safely
    df_compare = df_valid.pivot(index='run', columns='algorithm', values=['search_time', 'path_length', 'node_count', 'num_iterations'])

    # Drop any remaining NaNs (safety)
    # df_compare = df_compare.dropna()

    # Difference and summary
    df_diff = pd.DataFrame({
        'search_time': df_compare[('search_time', 'BRRT_Optimize')] - df_compare[('search_time', 'BRRT')],
        'path_length': df_compare[('path_length', 'BRRT_Optimize')] -  df_compare[('path_length', 'BRRT')],
        'node_count': df_compare[('node_count', 'BRRT_Optimize')] - df_compare[('node_count', 'BRRT')],
        'num_iterations': df_compare[('num_iterations', 'BRRT_Optimize')] - df_compare[('num_iterations', 'BRRT')],
    })

    return success_rate_brrt_opt, success_rate_brrt, df_diff
    # df_mean_diff = df_diff.mean()
    # print("=== Mean Metric Differences (BRRT_Optimize - BRRT) ===")
    # print(df_mean_diff)
    # # df_mean_diff = df_diff.mean()

    # # print("=== Mean Metric Differences (BRRT_Optimize - BRRT, Only if Both Success) ===")
    # print(df_compare)

def compare_algorithms_success(df,number_of_algorithms=2):

    # Filter only successful runs

    df_success = df[df['success'] == True]
    count_algo_success = df.groupby('algorithm').agg(
        success_rate=("success", "mean"))
    # Count how many algorithms succeeded per run
    success_counts = df_success.groupby('run')['algorithm'].nunique()

    # Keep only runs where both BRRT and BRRT_Optimize succeeded
    valid_runs = success_counts[success_counts == number_of_algorithms].index
    df_valid = df_success[df_success['run'].isin(valid_runs)]

    # Now pivot and compare safely
    df_compare = df_valid.pivot(index='run', columns='algorithm', values=['search_time', 'path_length', 'node_count', 'num_iterations'])


    return df_compare,count_algo_success


In [6]:
import glob
import pandas as pd
from itables import show

output_dir = '/home/huy/rrt/experiments/20250710/output-21/'
file_paths = glob.glob(output_dir + '*.json')

all_data_frames = []
columns_to_sum = ["search_time", "path_length", "node_count", "num_iterations","success"]

for file_path in file_paths:
    data, config = load_a_file(file_path)
    
    all_data_frames.append(data)
    
    print(f"\n=======================================================")
    print(f"--- Output for: {file_path.split('/')[-1]} ---")
    print(f"=======================================================")
    print("Config:", config)

    success_data = data[data["success"] == True]
    summed_data = success_data.groupby("algorithm")[columns_to_sum].sum()

    show(summed_data, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")

if all_data_frames:
    print("\n=======================================================")
    print("============= TOTAL SUM ACROSS ALL OUTPUTS =============")
    print("=======================================================")
    
    master_data = pd.concat(all_data_frames, ignore_index=True)
    
    master_success_data = master_data[master_data["success"] == True]
    master_summed = master_success_data.groupby("algorithm")[columns_to_sum].sum()
    
    show(master_summed, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
else:
    print("No JSON files found in the directory.")



--- Output for: brrt_0.50_0.5_3_6_4_1_9_2.json ---
Config: {'alpha': 6.0, 'beta': 4.0, 'epsilon': 0.5, 'gamma': 1.0, 'n_rays': 9, 'p1': 0.5, 'p2': 0.5, 'u_p': 3.0, 'weight_grade': 2}



--- Output for: brrt_2.00_0.5_3_1_7_1_36_2.json ---
Config: {'alpha': 1.0, 'beta': 7.0, 'epsilon': 2.0, 'gamma': 1.0, 'n_rays': 36, 'p1': 0.5, 'p2': 0.5, 'u_p': 3.0, 'weight_grade': 2}



--- Output for: brrt_0.50_0.3_0_10_1_7_9_1.json ---
Config: {'alpha': 10.0, 'beta': 1.0, 'epsilon': 0.5, 'gamma': 7.0, 'n_rays': 9, 'p1': 0.3, 'p2': 0.5, 'u_p': 0.0, 'weight_grade': 1}



--- Output for: brrt_10.00_0.5_1_1_0_0_18_1.json ---
Config: {'alpha': 1.0, 'beta': 0.0, 'epsilon': 10.0, 'gamma': 0.0, 'n_rays': 18, 'p1': 0.5, 'p2': 0.5, 'u_p': 1.0, 'weight_grade': 1}



--- Output for: brrt_10.00_0.3_12_3_0_5_18_2.json ---
Config: {'alpha': 3.0, 'beta': 0.0, 'epsilon': 10.0, 'gamma': 5.0, 'n_rays': 18, 'p1': 0.3, 'p2': 0.5, 'u_p': 12.0, 'weight_grade': 2}


...


--- Output for: brrt_10.00_0.5_12_1_1_0_9_2.json ---
Config: {'alpha': 1.0, 'beta': 1.0, 'epsilon': 10.0, 'gamma': 0.0, 'n_rays': 9, 'p1': 0.5, 'p2': 0.5, 'u_p': 12.0, 'weight_grade': 2}



--- Output for: brrt_1.00_0.5_1_6_0_0_18_1.json ---
Config: {'alpha': 6.0, 'beta': 0.0, 'epsilon': 1.0, 'gamma': 0.0, 'n_rays': 18, 'p1': 0.5, 'p2': 0.5, 'u_p': 1.0, 'weight_grade': 1}



--- Output for: brrt_10.00_0.3_12_0_0_7_18_1.json ---
Config: {'alpha': 0.0, 'beta': 0.0, 'epsilon': 10.0, 'gamma': 7.0, 'n_rays': 18, 'p1': 0.3, 'p2': 0.5, 'u_p': 12.0, 'weight_grade': 1}



--- Output for: brrt_1.00_0.8_0.5_1_4_7_36_1.json ---
Config: {'alpha': 1.0, 'beta': 4.0, 'epsilon': 1.0, 'gamma': 7.0, 'n_rays': 36, 'p1': 0.8, 'p2': 0.5, 'u_p': 0.5, 'weight_grade': 1}



============= TOTAL SUM ACROSS ALL OUTPUTS =============


In [ ]:


file_paths = glob.glob("/home/huy/rrt/experiments/20250709/output-10/*")
cout = 0
cout_success = 0
cout_path_length = 0
cout_node_count = 0
cout_iterations = 0
count_case1 = 0
count_case2 = 0
for file_path in file_paths:
    data,config = load_a_file(file_path)
    
    compared , cout_success = compare_algorithms_success(data,number_of_algorithms=3)
    unstacked = compared.mean().unstack()
    BRRT_rate = cout_success["success_rate"]["BRRT"]
    CASE1_rate = cout_success["success_rate"]["BRRT_Case1"]
    CASE2_rate = cout_success["success_rate"]["BRRT_Case2"]
    BRRT_search_time = unstacked["BRRT"]["search_time"]
    CASE1_search_time = unstacked["BRRT_Case1"]["search_time"]
    CASE2_search_time = unstacked["BRRT_Case2"]["search_time"]
    if CASE1_rate > BRRT_rate and CASE1_rate > CASE2_rate:
        count_case1 += 1
        if CASE1_search_time < BRRT_search_time:
            print("********************************************")
            print("BRRT_Case1 outperforms BRRT, BRRT_Case2")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        elif BRRT_search_time > CASE2_search_time and CASE2_rate > BRRT_rate:
            print("********************************************")
            print("BRRT_Case2 outperforms search time BRRT_Case1 but not success rate")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        # else:
        #     print("==========================================")
        #     print("BRRT_Case1 outperforms BRRT, BRRT_Case2 but with longer search time")
        #     print(config)
        #     print(unstacked)
        #     print(cout_success) 
        #     print("==========================================")

    elif CASE2_rate > BRRT_rate and CASE2_rate > CASE1_rate:
        count_case2 += 1
        if CASE2_search_time < BRRT_search_time:
            print("********************************************")
            print("BRRT_Case2 outperforms BRRT, BRRT_Case1")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        elif CASE1_search_time < BRRT_search_time and CASE1_rate > BRRT_rate:
            print("********************************************")
            print("BRRT_Case1 outperforms time BRRT_Case2 but not success rate")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        # else:
        #     print("==========================================")
        #     print("BRRT_Case2 outperforms BRRT, BRRT_Case1 but with longer search time")
        #     print(config)
        #     print(unstacked)
        #     print(cout_success)
        #     print("==========================================")

    
#     print(f"BRRT_rate: {BRRT_rate}, CASE1_rate: {CASE1_rate}, CASE2_rate: {CASE2_rate}")
    # if CASE1_rate > BRRT_rate:
    #     if CASE2_rate > CASE1_rate:
    #         count_case2 += 1
    #         print("==========================================")
    #         print("BRRT_Case2 outperforms BRRT, BRRT_Case1")
    #         print(config)
    #         print(unstacked)
    #         print(cout_success)
    #         print("==========================================")
    #     else:
    #         count_case1 += 1
    #         print("==========================================")
    #         print("BRRT_Case1 outperforms BRRT, BRRT_Case2")
    #         print(config)
    #         print(unstacked)
    #         print(cout_success)
    #         print("==========================================")
print(f"Total parameter settings where BRRT_Case1 best success: {count_case1}")           
print(f"Total parameter settings where BRRT_Case2 best success: {count_case2}")     
#     # if unstacked[]
#     # success_rate_brrt_opt, success_rate_brrt, diff =  compare_two_algorithms_success(data)
#     # success_rate = success_rate_brrt_opt - success_rate_brrt
#     # mean= diff.mean()
#     # if success_rate >= 0:
#     #     if mean["search_time"] < 0:
#     #         print ("----------------------------------------------")
#     #         print(success_rate_brrt_opt, success_rate_brrt)
#     #         print(f"Configuration: {config}")
#     #         print(f"Diff Success Rate: {success_rate:.2f}")
#     #         print(diff.mean())
#     #         print ("----------------------------------------------")
#     #         cout += 1
#     # if success_rate >= 0:
#     #     cout_success += 1
#     # if mean["path_length"] < 0:
#     #     cout_path_length += 1
#     # if mean["node_count"] < 0:
#     #     cout_node_count += 1
#     # if mean["num_iterations"] < 0:
#     #     cout_iterations += 1

#     break
# # print(f"Total parameter settings where BRRT_Optimize outperforms BRRT overall: {cout}")
# # print(f"Total parameter settings where BRRT_Optimize was success better: {cout_success}")
# # print(f"Total parameter settings where BRRT_Optimize achieved a shorter path length:  {cout_path_length}")
# # print(f"Total parameter settings where BRRT_Optimize used fewer nodes: {cout_node_count}")
# # print(f"Total parameter settings where BRRT_Optimize required fewer iterations:  {cout_iterations}")


In [ ]:
import glob
import json
file_paths = glob.glob("/home/huy/rrt/experiments/20250709/output-10/*")
data_agg = {}
data_summary_df = {}
for path in file_paths:
    with open(path, 'r') as f:
        content = json.load(f)
        sum_df, agg = summary_statistics(content)
        data_agg[path] = agg
        data_summary_df[path] = sum_df

In [ ]:
print(len(file_paths))

# Select the best

In [ ]:



def select_the_better_han(all_data:dict):
    best_all = 1000000
    best_all_file = None

    best_time = 1000000
    best_time_file = None

    best_path_length = 1000000
    best_path_length_file = None    

    best_node_count = 1000000
    best_node_count_file = None

    best_iterations = 1000000
    best_iterations_file = None
    for filepath, df in all_data.items():
        if (df.loc['BRRT_Optimize']["success_rate"] < df.loc['BRRT']["success_rate"]):
            continue
        mul = df.loc['BRRT_Optimize']/ df.loc['BRRT']
        percent = mul [['avg_search_time', 'avg_path_length','avg_node_count', 'avg_iterations']]
        # print(f"File: {filepath}",percent)
        sum_percent = percent.sum().sum()
        if (sum_percent < best_all):
            best_all_file = filepath
            best_all = sum_percent
        if (percent['avg_search_time'] < best_time):
            best_time_file = filepath
            best_time = percent['avg_search_time']
     
        if (percent['avg_path_length'] < best_path_length):
            best_path_length_file = filepath
            best_path_length = percent['avg_path_length']
        if (percent['avg_node_count'] < best_node_count):
            best_node_count_file = filepath
            best_node_count = percent['avg_node_count']
        if (percent['avg_iterations'] < best_iterations):
            best_iterations_file = filepath
            best_iterations = percent['avg_iterations']
    return {
        "best_all_file": best_all_file,
        "best_all": best_all,
        "best_time_file": best_time_file,
        "best_time": best_time,
        "best_path_length_file": best_path_length_file,
        "best_path_length": best_path_length,
        "best_node_count_file": best_node_count_file,
        "best_node_count": best_node_count,
        "best_iterations_file": best_iterations_file,
        "best_iterations": best_iterations
    }
summary = select_the_better_han(data_agg)
 
print(f"best_all_file File: {summary['best_all_file']}, Score: {summary['best_all']}")

load_a_data(summary["best_all_file"])

print(f"best_time_file File: {summary['best_time_file']}, Score: {summary['best_time']}")
load_a_data(summary["best_time_file"])
print(f"best_path_length_file File: {summary['best_path_length_file']}, Score: {summary['best_path_length']}")
load_a_data(summary["best_path_length_file"])
print(f"best_node_count_file File: {summary['best_node_count_file']}, Score: {summary['best_node_count']}")
load_a_data(summary["best_node_count_file"])
print(f"best_iterations_file File: {summary['best_iterations_file']}, Score: {summary['best_iterations']}")
load_a_data(summary["best_iterations_file"])    


In [ ]:
for index, row in best_param.iterrows():
    print(f"File: {row['parameter']}, BRRT_Optimize_score: {row['BRRT_Optimize_score']}, Diff: {row['Diff']}")
    show(data_agg[row['parameter']])
    display_data(data_summary_df[row['parameter']])
